# Mathematical Foundations for Large Language Models

**Purpose**: This notebook provides hands-on implementations of mathematical concepts underlying LLMs, including probability theory, optimization algorithms, backpropagation through transformers, scaling laws, and numerical stability considerations.

**Companion Theory**: This notebook works alongside the [Mathematical Foundations chapter](../index.md) which provides comprehensive theoretical background, derivations, and advanced mathematical concepts.

**Learning Path**: 
1. Read theory sections in the main chapter
2. Run corresponding notebook cells for hands-on practice  
3. Experiment with parameters to deepen understanding

**Dependencies**: PyTorch, NumPy, Matplotlib, SciPy

**Reproducibility**: All random seeds are set for deterministic outputs. This notebook runs on CPU and requires no special hardware.

**Primary Sources**: 
- Kingma & Ba, 2014 - "Adam: A Method for Stochastic Optimization" ([arXiv:1412.6980](https://arxiv.org/abs/1412.6980))
- Kaplan et al., 2020 - "Scaling Laws for Neural Language Models" ([arXiv:2001.08361](https://arxiv.org/abs/2001.08361))
- Hoffmann et al., 2022 - "Training Compute-Optimal Large Language Models" ([arXiv:2203.15556](https://arxiv.org/abs/2203.15556))
- Ba et al., 2016 - "Layer Normalization" ([arXiv:1607.06450](https://arxiv.org/abs/1607.06450))

This notebook demonstrates the mathematical principles that enable effective training and scaling of large language models.

## 1. Setup and Imports

In [ ]:
# Install required packages if not already installed
import subprocess
import sys

def install_if_missing(package):
    """Install package if not already available."""
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])

# Install PyTorch CPU version
packages = ['torch', 'torchvision', 'torchaudio', '--index-url', 'https://download.pytorch.org/whl/cpu']
try:
    import torch
    print("PyTorch already installed")
except ImportError:
    print("Installing PyTorch CPU version...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + packages)

# Install other packages
other_packages = ['scipy']
for pkg in other_packages:
    install_if_missing(pkg)

print("All packages installed successfully!")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
from typing import Optional, Tuple, List
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Configure plotting
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10

print("=" * 60)
print("Environment Setup Complete")
print("=" * 60)
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"Random seed: {SEED}")
print("=" * 60)

## 2. Probability and Information Theory

Understanding probability distributions and information theory is fundamental to language modeling. This section explores entropy, cross-entropy, KL divergence, and their applications in LLMs.

**Theory Reference**: See Chapter 2, Section 1 for complete mathematical derivations and theoretical foundations.

In [ ]:
def entropy(probs: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """
    Compute Shannon entropy of probability distribution.
    
    H(X) = -∑ P(x) log P(x)
    
    Args:
        probs: Probability distribution [batch_size, num_classes]
        eps: Small constant for numerical stability
    
    Returns:
        Entropy values [batch_size]
    """
    return -torch.sum(probs * torch.log(probs + eps), dim=-1)


def cross_entropy_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """
    Compute cross-entropy loss between predictions and targets.
    
    H(P, Q) = -∑ P(x) log Q(x)
    
    Args:
        logits: Model outputs [batch_size, vocab_size]
        targets: True token indices [batch_size]
    
    Returns:
        Cross-entropy loss (scalar)
    """
    log_probs = F.log_softmax(logits, dim=-1)
    return F.nll_loss(log_probs, targets)


def kl_divergence(p: torch.Tensor, q: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """
    Compute Kullback-Leibler divergence between distributions.
    
    D_KL(P || Q) = ∑ P(x) log(P(x) / Q(x))
    
    Args:
        p: True distribution [batch_size, num_classes]
        q: Approximate distribution [batch_size, num_classes]
        eps: Small constant for numerical stability
    
    Returns:
        KL divergence values [batch_size]
    """
    return torch.sum(p * torch.log((p + eps) / (q + eps)), dim=-1)


def perplexity(loss: torch.Tensor) -> torch.Tensor:
    """
    Convert cross-entropy loss to perplexity.
    
    Perplexity = exp(H(P, Q))
    
    Args:
        loss: Cross-entropy loss
    
    Returns:
        Perplexity value
    """
    return torch.exp(loss)


# Demonstration: Compare different distributions
print("Probability and Information Theory Demonstration")
print("=" * 60)

# Create sample distributions
vocab_size = 10
batch_size = 3

# Uniform distribution (high entropy)
uniform_dist = torch.ones(batch_size, vocab_size) / vocab_size

# Peaked distribution (low entropy)
peaked_dist = torch.zeros(batch_size, vocab_size)
peaked_dist[:, 0] = 0.9
peaked_dist[:, 1:] = 0.1 / (vocab_size - 1)

# Slightly uncertain distribution (medium entropy)
uncertain_dist = torch.zeros(batch_size, vocab_size)
uncertain_dist[:, 0] = 0.5
uncertain_dist[:, 1] = 0.3
uncertain_dist[:, 2:] = 0.2 / (vocab_size - 2)

# Compute entropies
uniform_entropy = entropy(uniform_dist)
peaked_entropy = entropy(peaked_dist)
uncertain_entropy = entropy(uncertain_dist)

print("\nEntropy Analysis:")
print(f"Uniform distribution entropy: {uniform_entropy[0]:.4f} bits")
print(f"Peaked distribution entropy: {peaked_entropy[0]:.4f} bits")
print(f"Uncertain distribution entropy: {uncertain_entropy[0]:.4f} bits")
print(f"Maximum possible entropy (log {vocab_size}): {math.log(vocab_size):.4f} bits")

# Compute KL divergences
kl_uniform_peaked = kl_divergence(uniform_dist, peaked_dist)
kl_peaked_uniform = kl_divergence(peaked_dist, uniform_dist)

print("\nKL Divergence (asymmetric):")
print(f"D_KL(uniform || peaked): {kl_uniform_peaked[0]:.4f}")
print(f"D_KL(peaked || uniform): {kl_peaked_uniform[0]:.4f}")
print("Note: KL divergence is not symmetric!")

# Simulate language model prediction
logits = torch.randn(batch_size, vocab_size)
targets = torch.randint(0, vocab_size, (batch_size,))
ce_loss = cross_entropy_loss(logits, targets)
ppl = perplexity(ce_loss)

print("\nLanguage Model Metrics:")
print(f"Cross-entropy loss: {ce_loss:.4f}")
print(f"Perplexity: {ppl:.4f}")
print("=" * 60)

### Visualization: Entropy vs Distribution Shape

In [ ]:
# Create distributions with varying concentration
alphas = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
vocab_size = 20

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, alpha in enumerate(alphas):
    # Create distribution with concentration parameter alpha
    probs = torch.softmax(torch.randn(vocab_size) * alpha, dim=-1)
    ent = entropy(probs.unsqueeze(0))[0].item()
    
    # Plot
    axes[idx].bar(range(vocab_size), probs.numpy())
    axes[idx].set_title(f'α={alpha}, Entropy={ent:.3f}')
    axes[idx].set_xlabel('Token')
    axes[idx].set_ylabel('Probability')
    axes[idx].set_ylim(0, 1)

plt.tight_layout()
plt.suptitle('Entropy vs Distribution Concentration', y=1.02, fontsize=14, fontweight='bold')
plt.show()

print("Observation: As distribution becomes more peaked (higher α), entropy decreases.")
print("Lower entropy means more predictable/certain predictions.")

## 3. Optimization Algorithms

This section implements and compares fundamental optimization algorithms used in LLM training: SGD, Momentum, and Adam.

**Theory Reference**: See Chapter 2, Section 2 for mathematical derivations and convergence analysis.

In [ ]:
class SGDOptimizer:
    """Stochastic Gradient Descent with optional momentum."""
    
    def __init__(self, params: List[torch.Tensor], lr: float = 0.01, momentum: float = 0.0):
        """
        Initialize SGD optimizer.
        
        Args:
            params: Model parameters to optimize
            lr: Learning rate
            momentum: Momentum coefficient (0 for vanilla SGD)
        """
        self.params = list(params)
        self.lr = lr
        self.momentum = momentum
        self.velocities = [torch.zeros_like(p) for p in self.params]
        
    def step(self):
        """Perform single optimization step."""
        with torch.no_grad():
            for i, param in enumerate(self.params):
                if param.grad is None:
                    continue
                
                # Update velocity: v = β*v + ∇L
                self.velocities[i] = self.momentum * self.velocities[i] + param.grad
                
                # Update parameter: θ = θ - η*v
                param -= self.lr * self.velocities[i]
    
    def zero_grad(self):
        """Zero out gradients."""
        for param in self.params:
            if param.grad is not None:
                param.grad.zero_()


class AdamOptimizer:
    """Adam optimizer with bias correction."""
    
    def __init__(
        self,
        params: List[torch.Tensor],
        lr: float = 0.001,
        betas: Tuple[float, float] = (0.9, 0.999),
        eps: float = 1e-8,
        weight_decay: float = 0.0
    ):
        """
        Initialize Adam optimizer.
        
        Args:
            params: Model parameters to optimize
            lr: Learning rate
            betas: Coefficients for computing running averages (β₁, β₂)
            eps: Term added for numerical stability
            weight_decay: L2 penalty coefficient
        """
        self.params = list(params)
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.weight_decay = weight_decay
        
        # Initialize moment estimates
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        self.step_count = 0
    
    def step(self):
        """Perform single optimization step."""
        self.step_count += 1
        
        with torch.no_grad():
            for i, param in enumerate(self.params):
                if param.grad is None:
                    continue
                
                grad = param.grad
                
                # Add weight decay (L2 regularization)
                if self.weight_decay > 0:
                    grad = grad + self.weight_decay * param
                
                # Update biased first moment estimate: m = β₁*m + (1-β₁)*g
                self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * grad
                
                # Update biased second moment estimate: v = β₂*v + (1-β₂)*g²
                self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * grad.pow(2)
                
                # Compute bias-corrected moment estimates
                m_hat = self.m[i] / (1 - self.beta1 ** self.step_count)
                v_hat = self.v[i] / (1 - self.beta2 ** self.step_count)
                
                # Update parameters: θ = θ - η * m̂ / (√v̂ + ε)
                param -= self.lr * m_hat / (torch.sqrt(v_hat) + self.eps)
    
    def zero_grad(self):
        """Zero out gradients."""
        for param in self.params:
            if param.grad is not None:
                param.grad.zero_()


print("Optimizer implementations ready: SGD, SGD+Momentum, Adam")
print("Next: Compare optimization trajectories on a toy problem")

### Optimization Comparison: Rosenbrock Function

We'll compare different optimizers on the Rosenbrock function, a classic optimization benchmark with a narrow valley making it challenging for gradient descent.

In [ ]:
def rosenbrock(x: torch.Tensor, y: torch.Tensor, a: float = 1.0, b: float = 100.0) -> torch.Tensor:
    """
    Rosenbrock function: f(x,y) = (a-x)² + b(y-x²)²
    Global minimum at (a, a²) with f(a, a²) = 0
    """
    return (a - x)**2 + b * (y - x**2)**2


def optimize_rosenbrock(optimizer_class, optimizer_kwargs, num_steps=1000):
    """Run optimization and track trajectory."""
    # Initialize parameters far from optimum
    x = torch.tensor([-1.0], requires_grad=True)
    y = torch.tensor([-1.0], requires_grad=True)
    
    optimizer = optimizer_class([x, y], **optimizer_kwargs)
    
    trajectory_x = []
    trajectory_y = []
    losses = []
    
    for step in range(num_steps):
        optimizer.zero_grad()
        
        # Compute loss
        loss = rosenbrock(x, y)
        loss.backward()
        
        # Record trajectory
        trajectory_x.append(x.item())
        trajectory_y.append(y.item())
        losses.append(loss.item())
        
        # Update parameters
        optimizer.step()
    
    return np.array(trajectory_x), np.array(trajectory_y), np.array(losses)


# Compare different optimizers
optimizers_config = [
    ("SGD", SGDOptimizer, {"lr": 0.001, "momentum": 0.0}),
    ("SGD+Momentum", SGDOptimizer, {"lr": 0.001, "momentum": 0.9}),
    ("Adam", AdamOptimizer, {"lr": 0.01}),
]

results = {}
for name, opt_class, opt_kwargs in optimizers_config:
    print(f"Running {name}...")
    traj_x, traj_y, losses = optimize_rosenbrock(opt_class, opt_kwargs)
    results[name] = (traj_x, traj_y, losses)

print("Optimization runs complete!")

In [ ]:
# Visualize optimization trajectories
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Create contour plot of Rosenbrock function
x_range = np.linspace(-2, 2, 100)
y_range = np.linspace(-1, 3, 100)
X, Y = np.meshgrid(x_range, y_range)
Z = (1 - X)**2 + 100 * (Y - X**2)**2

# Plot 1: Trajectories on contour
ax1 = axes[0]
contour = ax1.contour(X, Y, Z, levels=np.logspace(-1, 3, 20), cmap='viridis', alpha=0.6)
ax1.clabel(contour, inline=True, fontsize=8)

for name, (traj_x, traj_y, _) in results.items():
    ax1.plot(traj_x, traj_y, 'o-', label=name, markersize=2, linewidth=1.5, alpha=0.7)

ax1.plot(1, 1, 'r*', markersize=20, label='Optimum')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_title('Optimization Trajectories on Rosenbrock Function')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Loss curves
ax2 = axes[1]
for name, (_, _, losses) in results.items():
    ax2.plot(losses, label=name, linewidth=2)

ax2.set_xlabel('Iteration')
ax2.set_ylabel('Loss')
ax2.set_title('Loss Convergence')
ax2.set_yscale('log')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nFinal losses:")
for name, (_, _, losses) in results.items():
    print(f"{name}: {losses[-1]:.6f}")

## 4. Learning Rate Scheduling

Learning rate schedules are critical for stable training. This section implements common schedules used in LLM training.

**Theory Reference**: See Chapter 2, Section 2.2 for mathematical analysis of different schedules.

In [ ]:
class LearningRateScheduler:
    """Collection of learning rate schedules for LLM training."""
    
    @staticmethod
    def linear_warmup_cosine_decay(
        step: int,
        warmup_steps: int,
        max_steps: int,
        max_lr: float,
        min_lr: float
    ) -> float:
        """
        Linear warmup followed by cosine decay.
        
        Widely used in transformer training (BERT, GPT, etc.)
        
        Args:
            step: Current training step
            warmup_steps: Number of warmup steps
            max_steps: Total training steps
            max_lr: Maximum learning rate
            min_lr: Minimum learning rate
        
        Returns:
            Learning rate for current step
        """
        if step < warmup_steps:
            # Linear warmup: lr = max_lr * (step / warmup_steps)
            return max_lr * (step / warmup_steps)
        else:
            # Cosine decay
            progress = (step - warmup_steps) / (max_steps - warmup_steps)
            return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))
    
    @staticmethod
    def inverse_sqrt_decay(
        step: int,
        warmup_steps: int,
        max_lr: float
    ) -> float:
        """
        Inverse square root decay after warmup.
        
        Used in original Transformer paper.
        
        Args:
            step: Current training step
            warmup_steps: Number of warmup steps
            max_lr: Maximum learning rate
        
        Returns:
            Learning rate for current step
        """
        if step < warmup_steps:
            return max_lr * (step / warmup_steps)
        else:
            return max_lr * math.sqrt(warmup_steps / step)
    
    @staticmethod
    def constant_with_warmup(
        step: int,
        warmup_steps: int,
        lr: float
    ) -> float:
        """
        Linear warmup to constant learning rate.
        
        Args:
            step: Current training step
            warmup_steps: Number of warmup steps
            lr: Target learning rate
        
        Returns:
            Learning rate for current step
        """
        if step < warmup_steps:
            return lr * (step / warmup_steps)
        else:
            return lr


# Visualize different schedules
max_steps = 10000
warmup_steps = 1000
max_lr = 1e-3
min_lr = 1e-5

steps = np.arange(0, max_steps)
schedules = {
    'Linear Warmup + Cosine Decay': [
        LearningRateScheduler.linear_warmup_cosine_decay(
            s, warmup_steps, max_steps, max_lr, min_lr
        ) for s in steps
    ],
    'Inverse Sqrt Decay': [
        LearningRateScheduler.inverse_sqrt_decay(s, warmup_steps, max_lr) 
        for s in steps
    ],
    'Constant with Warmup': [
        LearningRateScheduler.constant_with_warmup(s, warmup_steps, max_lr)
        for s in steps
    ],
}

plt.figure(figsize=(12, 6))
for name, lrs in schedules.items():
    plt.plot(steps, lrs, label=name, linewidth=2)

plt.axvline(x=warmup_steps, color='red', linestyle='--', alpha=0.5, label='Warmup End')
plt.xlabel('Training Step')
plt.ylabel('Learning Rate')
plt.title('Learning Rate Schedules for LLM Training')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.show()

print(f"Warmup steps: {warmup_steps}")
print(f"Total steps: {max_steps}")
print(f"Max learning rate: {max_lr}")
print(f"Min learning rate: {min_lr}")

## 5. Backpropagation Through Transformers

Understanding gradient flow through transformer components is essential for effective training. This section implements and analyzes attention mechanisms and layer normalization.

**Theory Reference**: See Chapter 2, Section 3 for complete mathematical derivations.

In [ ]:
class ScaledDotProductAttention(nn.Module):
    """
    Scaled dot-product attention mechanism.
    
    Attention(Q, K, V) = softmax(QK^T / √d_k)V
    """
    
    def __init__(self, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
    
    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Compute scaled dot-product attention.
        
        Args:
            query: Query tensor [batch, heads, seq_len, d_k]
            key: Key tensor [batch, heads, seq_len, d_k]
            value: Value tensor [batch, heads, seq_len, d_v]
            mask: Optional attention mask
        
        Returns:
            output: Attention output [batch, heads, seq_len, d_v]
            attention_weights: Attention weights [batch, heads, seq_len, seq_len]
        """
        d_k = query.size(-1)
        
        # Compute attention scores: QK^T / √d_k
        scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
        
        # Apply mask if provided
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        # Apply softmax to get attention weights
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        # Apply attention weights to values
        output = torch.matmul(attention_weights, value)
        
        return output, attention_weights


def analyze_attention_gradients(
    batch_size: int = 2,
    seq_len: int = 8,
    d_model: int = 64
):
    """
    Analyze gradient flow through attention mechanism.
    """
    # Create attention module
    attention = ScaledDotProductAttention(dropout=0.0)
    
    # Generate random Q, K, V with gradients enabled
    query = torch.randn(batch_size, 1, seq_len, d_model, requires_grad=True)
    key = torch.randn(batch_size, 1, seq_len, d_model, requires_grad=True)
    value = torch.randn(batch_size, 1, seq_len, d_model, requires_grad=True)
    
    # Forward pass
    output, attn_weights = attention(query, key, value)
    
    # Compute dummy loss (sum of outputs)
    loss = output.sum()
    loss.backward()
    
    # Analyze gradient magnitudes
    print("Gradient Flow Analysis Through Attention")
    print("=" * 60)
    print(f"Query gradient norm: {query.grad.norm().item():.4f}")
    print(f"Key gradient norm: {key.grad.norm().item():.4f}")
    print(f"Value gradient norm: {value.grad.norm().item():.4f}")
    print(f"\nAttention weights shape: {attn_weights.shape}")
    print(f"Output shape: {output.shape}")
    print("=" * 60)
    
    return attn_weights[0, 0].detach().numpy()

# Run analysis
attn_matrix = analyze_attention_gradients()

# Visualize attention weights
plt.figure(figsize=(8, 6))
sns.heatmap(attn_matrix, cmap='viridis', annot=True, fmt='.3f', cbar=True)
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.title('Attention Weights Matrix')
plt.tight_layout()
plt.show()

### Layer Normalization and Gradient Flow

In [ ]:
class LayerNorm(nn.Module):
    """
    Layer normalization with explicit gradient computation.
    
    y = γ * (x - μ) / σ + β
    
    where μ and σ are computed over the feature dimension.
    """
    
    def __init__(self, normalized_shape: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(normalized_shape))
        self.beta = nn.Parameter(torch.zeros(normalized_shape))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Apply layer normalization.
        
        Args:
            x: Input tensor [..., normalized_shape]
        
        Returns:
            Normalized tensor with same shape as input
        """
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True, unbiased=False)
        
        # Normalize
        x_norm = (x - mean) / (std + self.eps)
        
        # Scale and shift
        return self.gamma * x_norm + self.beta


def compare_gradient_flow():
    """
    Compare gradient flow with and without layer normalization.
    """
    batch_size = 4
    seq_len = 16
    d_model = 128
    num_layers = 12
    
    print("Gradient Flow Comparison: With vs Without Layer Normalization")
    print("=" * 60)
    
    # Test without layer normalization
    x = torch.randn(batch_size, seq_len, d_model, requires_grad=True)
    
    # Simulate deep network without normalization
    activations_no_norm = [x]
    for _ in range(num_layers):
        # Simple linear transformation
        W = torch.randn(d_model, d_model, requires_grad=True) * 0.02
        x = torch.matmul(x, W)
        activations_no_norm.append(x)
    
    loss_no_norm = x.sum()
    loss_no_norm.backward()
    
    grad_norms_no_norm = [act.grad.norm().item() if act.grad is not None else 0 
                           for act in activations_no_norm]
    
    # Test with layer normalization
    x = torch.randn(batch_size, seq_len, d_model, requires_grad=True)
    
    # Simulate deep network with normalization
    activations_with_norm = [x]
    layer_norms = [LayerNorm(d_model) for _ in range(num_layers)]
    
    for ln in layer_norms:
        W = torch.randn(d_model, d_model, requires_grad=True) * 0.02
        x = torch.matmul(x, W)
        x = ln(x)
        activations_with_norm.append(x)
    
    loss_with_norm = x.sum()
    loss_with_norm.backward()
    
    grad_norms_with_norm = [act.grad.norm().item() if act.grad is not None else 0 
                             for act in activations_with_norm]
    
    # Visualize results
    layers = list(range(num_layers + 1))
    
    plt.figure(figsize=(12, 6))
    plt.plot(layers, grad_norms_no_norm, 'o-', label='Without LayerNorm', linewidth=2)
    plt.plot(layers, grad_norms_with_norm, 's-', label='With LayerNorm', linewidth=2)
    plt.xlabel('Layer Depth')
    plt.ylabel('Gradient Norm')
    plt.title('Gradient Flow: Impact of Layer Normalization')
    plt.yscale('log')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"Final gradient norm without LayerNorm: {grad_norms_no_norm[-1]:.6f}")
    print(f"Final gradient norm with LayerNorm: {grad_norms_with_norm[-1]:.6f}")
    
    # Avoid division by zero
    if grad_norms_no_norm[-1] > 1e-10:
        improvement = grad_norms_with_norm[-1] / grad_norms_no_norm[-1]
        print(f"Improvement factor: {improvement:.2f}x")
    else:
        print("Improvement factor: Cannot compute (gradient vanished)")
    
    print("=" * 60)

compare_gradient_flow()

## 6. Scaling Laws

Scaling laws describe predictable relationships between model size, data, compute, and performance. This section simulates and visualizes these relationships.

**Theory Reference**: See Chapter 2, Section 4 for complete mathematical treatment of Kaplan and Chinchilla scaling laws.

In [ ]:
def kaplan_scaling_law(
    N: np.ndarray,
    N_c: float = 8.8e13,
    alpha_N: float = 0.076
) -> np.ndarray:
    """
    Kaplan et al. parameter scaling law.
    
    L(N) = (N_c / N)^α_N
    
    Args:
        N: Number of parameters
        N_c: Critical scale parameter
        alpha_N: Scaling exponent
    
    Returns:
        Loss values
    """
    return (N_c / N) ** alpha_N


def data_scaling_law(
    D: np.ndarray,
    D_c: float = 5.4e13,
    alpha_D: float = 0.095
) -> np.ndarray:
    """
    Data scaling law.
    
    L(D) = (D_c / D)^α_D
    
    Args:
        D: Dataset size (tokens)
        D_c: Critical scale parameter
        alpha_D: Scaling exponent
    
    Returns:
        Loss values
    """
    return (D_c / D) ** alpha_D


def compute_scaling_law(
    C: np.ndarray,
    C_c: float = 1e10,
    alpha_C: float = 0.05
) -> np.ndarray:
    """
    Compute scaling law.
    
    L(C) = (C_c / C)^α_C
    
    Args:
        C: Training compute (FLOPs)
        C_c: Critical scale parameter
        alpha_C: Scaling exponent
    
    Returns:
        Loss values
    """
    return (C_c / C) ** alpha_C


def chinchilla_optimal_allocation(
    compute_budget: float,
    a: float = 0.5,
    b: float = 0.5
) -> Tuple[float, float]:
    """
    Compute optimal model size and data size for given compute budget.
    
    Based on Chinchilla scaling laws (Hoffmann et al., 2022).
    
    N* ∝ C^a
    D* ∝ C^b
    
    Args:
        compute_budget: Available compute (FLOPs)
        a: Model size scaling exponent
        b: Data size scaling exponent
    
    Returns:
        Optimal (model_size, data_size)
    """
    # Simplified allocation (actual constants would be fitted to empirical data)
    model_size = compute_budget ** a / 1e6  # Rough approximation
    data_size = compute_budget ** b / 1e3   # Rough approximation
    
    return model_size, data_size


# Visualize scaling laws
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Parameter scaling
N_range = np.logspace(6, 12, 100)  # 1M to 1T parameters
losses_N = kaplan_scaling_law(N_range)

axes[0, 0].loglog(N_range, losses_N, linewidth=2)
axes[0, 0].set_xlabel('Number of Parameters (N)')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Parameter Scaling Law: L(N) = (N_c/N)^α')
axes[0, 0].grid(True, alpha=0.3)

# Mark notable model sizes
notable_sizes = {
    'GPT-2': 1.5e9,
    'GPT-3': 175e9,
    'LLaMA-70B': 70e9,
}
for model_name, size in notable_sizes.items():
    if N_range.min() <= size <= N_range.max():
        loss = kaplan_scaling_law(np.array([size]))[0]
        axes[0, 0].plot(size, loss, 'ro', markersize=8)
        axes[0, 0].text(size, loss * 1.2, model_name, ha='center')

# Data scaling
D_range = np.logspace(9, 13, 100)  # 1B to 10T tokens
losses_D = data_scaling_law(D_range)

axes[0, 1].loglog(D_range, losses_D, linewidth=2, color='orange')
axes[0, 1].set_xlabel('Dataset Size (tokens)')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].set_title('Data Scaling Law: L(D) = (D_c/D)^α')
axes[0, 1].grid(True, alpha=0.3)

# Compute scaling
C_range = np.logspace(18, 24, 100)  # 1e18 to 1e24 FLOPs
losses_C = compute_scaling_law(C_range)

axes[1, 0].loglog(C_range, losses_C, linewidth=2, color='green')
axes[1, 0].set_xlabel('Training Compute (FLOPs)')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].set_title('Compute Scaling Law: L(C) = (C_c/C)^α')
axes[1, 0].grid(True, alpha=0.3)

# Chinchilla optimal allocation
compute_budgets = np.logspace(18, 24, 50)
optimal_N = []
optimal_D = []

for C in compute_budgets:
    N_opt, D_opt = chinchilla_optimal_allocation(C)
    optimal_N.append(N_opt)
    optimal_D.append(D_opt)

optimal_N = np.array(optimal_N)
optimal_D = np.array(optimal_D)

axes[1, 1].loglog(compute_budgets, optimal_N, label='Optimal Model Size', linewidth=2)
axes[1, 1].loglog(compute_budgets, optimal_D, label='Optimal Data Size', linewidth=2)
axes[1, 1].set_xlabel('Training Compute (FLOPs)')
axes[1, 1].set_ylabel('Size')
axes[1, 1].set_title('Chinchilla: Compute-Optimal Allocation')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Scaling Laws Summary")
print("=" * 60)
print("Parameter Scaling: L ∝ N^(-0.076)")
print("Data Scaling: L ∝ D^(-0.095)")
print("Compute Scaling: L ∝ C^(-0.05)")
print("\nChinchilla Result: For optimal performance, scale")
print("model size and data size equally with compute (N* ∝ C^0.5, D* ∝ C^0.5)")
print("=" * 60)

### Memory and Compute Requirements

In [ ]:
def estimate_training_requirements(
    num_params_M: float,
    seq_length: int = 2048,
    batch_size: int = 1,
    num_layers: int = 32,
    d_model: int = 4096,
    vocab_size: int = 50000,
    precision: str = 'fp16'
) -> dict:
    """
    Estimate memory and compute requirements for transformer training.
    
    Args:
        num_params_M: Model size in millions of parameters
        seq_length: Sequence length
        batch_size: Training batch size
        num_layers: Number of transformer layers
        d_model: Model dimension
        vocab_size: Vocabulary size
        precision: 'fp16' or 'fp32'
    
    Returns:
        Dictionary with memory and compute estimates
    """
    bytes_per_param = 2 if precision == 'fp16' else 4
    
    # Parameter memory
    param_memory_GB = (num_params_M * 1e6 * bytes_per_param) / 1e9
    
    # Optimizer state memory (Adam: 2x parameters for m and v)
    optimizer_memory_GB = param_memory_GB * 2
    
    # Activation memory (rough estimate)
    activation_memory_GB = (batch_size * seq_length * d_model * num_layers * bytes_per_param) / 1e9
    
    # Gradient memory (same as parameters)
    gradient_memory_GB = param_memory_GB
    
    # Total memory
    total_memory_GB = param_memory_GB + optimizer_memory_GB + activation_memory_GB + gradient_memory_GB
    
    # Compute per forward pass (in TFLOPs)
    # Simplified: 2 * params * batch_size * seq_length
    compute_per_forward_TFLOPS = (2 * num_params_M * 1e6 * batch_size * seq_length) / 1e12
    
    # Total compute (forward + backward ≈ 3x forward)
    compute_per_step_TFLOPS = compute_per_forward_TFLOPS * 3
    
    return {
        'parameter_memory_GB': param_memory_GB,
        'optimizer_memory_GB': optimizer_memory_GB,
        'activation_memory_GB': activation_memory_GB,
        'gradient_memory_GB': gradient_memory_GB,
        'total_memory_GB': total_memory_GB,
        'compute_per_step_TFLOPS': compute_per_step_TFLOPS,
    }


# Analyze different model sizes
model_configs = [
    ('GPT-2 Small', 117, 12, 768, 2048),
    ('GPT-2 Large', 774, 36, 1280, 2048),
    ('GPT-3 Small', 125, 12, 768, 2048),
    ('GPT-3 Medium', 350, 24, 1024, 2048),
    ('GPT-3 Large', 1300, 24, 2048, 2048),
    ('GPT-3 XL', 2700, 32, 2560, 2048),
    ('GPT-3 175B', 175000, 96, 12288, 2048),
]

print("Memory and Compute Requirements Analysis")
print("=" * 80)
print(f"{'Model':<20} {'Params (M)':<15} {'Memory (GB)':<15} {'Compute (TFLOP/step)':<20}")
print("=" * 80)

memory_requirements = []
compute_requirements = []
model_names = []

for name, params_M, layers, d_model, seq_len in model_configs:
    reqs = estimate_training_requirements(
        num_params_M=params_M,
        num_layers=layers,
        d_model=d_model,
        seq_length=seq_len,
        batch_size=8,
        precision='fp16'
    )
    
    print(f"{name:<20} {params_M:<15.0f} {reqs['total_memory_GB']:<15.1f} {reqs['compute_per_step_TFLOPS']:<20.2f}")
    
    memory_requirements.append(reqs['total_memory_GB'])
    compute_requirements.append(reqs['compute_per_step_TFLOPS'])
    model_names.append(name)

print("=" * 80)

# Visualize scaling
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

params = [m[1] for m in model_configs]

ax1.loglog(params, memory_requirements, 'o-', linewidth=2, markersize=8)
ax1.set_xlabel('Model Size (M parameters)')
ax1.set_ylabel('Total Memory (GB)')
ax1.set_title('Memory Requirements vs Model Size')
ax1.grid(True, alpha=0.3)

ax2.loglog(params, compute_requirements, 's-', linewidth=2, markersize=8, color='orange')
ax2.set_xlabel('Model Size (M parameters)')
ax2.set_ylabel('Compute (TFLOP/step)')
ax2.set_title('Compute Requirements vs Model Size')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Numerical Stability

Numerical precision and stability are critical for training large models. This section explores floating-point precision, initialization strategies, and gradient clipping.

**Theory Reference**: See Chapter 2, Section 5 for complete treatment of numerical stability considerations.

In [ ]:
def demonstrate_fp16_underflow():
    """
    Demonstrate gradient underflow in FP16 precision.
    """
    print("Numerical Precision Demonstration: FP16 vs FP32")
    print("=" * 60)
    
    # Small gradient that might underflow in FP16
    grad_fp32 = torch.tensor([1e-8], dtype=torch.float32)
    grad_fp16 = torch.tensor([1e-8], dtype=torch.float16)
    
    print(f"Original value (FP32): {grad_fp32.item():.2e}")
    print(f"Converted to FP16: {grad_fp16.item():.2e}")
    print(f"Loss of precision: {abs(grad_fp32.item() - grad_fp16.item()) / grad_fp32.item() * 100:.2f}%")
    
    # Show FP16 range limits
    print(f"\nFP16 representable range:")
    print(f"  Maximum: {torch.finfo(torch.float16).max:.2e}")
    print(f"  Minimum (positive): {torch.finfo(torch.float16).tiny:.2e}")
    print(f"  Epsilon: {torch.finfo(torch.float16).eps:.2e}")
    
    print(f"\nFP32 representable range:")
    print(f"  Maximum: {torch.finfo(torch.float32).max:.2e}")
    print(f"  Minimum (positive): {torch.finfo(torch.float32).tiny:.2e}")
    print(f"  Epsilon: {torch.finfo(torch.float32).eps:.2e}")
    print("=" * 60)


def gradient_clipping_demo():
    """
    Demonstrate gradient clipping for stability.
    """
    print("\nGradient Clipping Demonstration")
    print("=" * 60)
    
    # Create parameters with large gradients
    params = [torch.randn(100, 100, requires_grad=True) for _ in range(3)]
    
    # Simulate large gradients (exploding gradient scenario)
    for p in params:
        p.grad = torch.randn_like(p) * 100  # Very large gradients
    
    # Compute gradient norm before clipping
    total_norm_before = torch.sqrt(sum(p.grad.norm()**2 for p in params))
    
    # Apply gradient clipping
    max_norm = 1.0
    torch.nn.utils.clip_grad_norm_(params, max_norm)
    
    # Compute gradient norm after clipping
    total_norm_after = torch.sqrt(sum(p.grad.norm()**2 for p in params))
    
    print(f"Gradient norm before clipping: {total_norm_before.item():.2f}")
    print(f"Gradient norm after clipping: {total_norm_after.item():.2f}")
    print(f"Max norm threshold: {max_norm}")
    print(f"Clipping factor: {max_norm / total_norm_before.item():.4f}")
    print("=" * 60)


def initialization_comparison():
    """
    Compare different initialization strategies.
    """
    print("\nInitialization Strategies Comparison")
    print("=" * 60)
    
    n_in = 512
    n_out = 512
    num_samples = 10000
    
    # Xavier/Glorot initialization
    xavier_weight = torch.randn(n_in, n_out) * math.sqrt(2.0 / (n_in + n_out))
    
    # He initialization
    he_weight = torch.randn(n_in, n_out) * math.sqrt(2.0 / n_in)
    
    # Standard initialization
    std_weight = torch.randn(n_in, n_out) * 0.01
    
    # Analyze distributions
    print("Weight Distribution Statistics:")
    print(f"\nXavier Initialization:")
    print(f"  Mean: {xavier_weight.mean():.6f}")
    print(f"  Std:  {xavier_weight.std():.6f}")
    print(f"  Theoretical std: {math.sqrt(2.0 / (n_in + n_out)):.6f}")
    
    print(f"\nHe Initialization:")
    print(f"  Mean: {he_weight.mean():.6f}")
    print(f"  Std:  {he_weight.std():.6f}")
    print(f"  Theoretical std: {math.sqrt(2.0 / n_in):.6f}")
    
    print(f"\nStandard Initialization (0.01):")
    print(f"  Mean: {std_weight.mean():.6f}")
    print(f"  Std:  {std_weight.std():.6f}")
    
    # Visualize distributions
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].hist(xavier_weight.flatten().numpy(), bins=50, alpha=0.7, edgecolor='black')
    axes[0].set_title('Xavier Initialization')
    axes[0].set_xlabel('Weight Value')
    axes[0].set_ylabel('Frequency')
    
    axes[1].hist(he_weight.flatten().numpy(), bins=50, alpha=0.7, edgecolor='black', color='orange')
    axes[1].set_title('He Initialization')
    axes[1].set_xlabel('Weight Value')
    
    axes[2].hist(std_weight.flatten().numpy(), bins=50, alpha=0.7, edgecolor='black', color='green')
    axes[2].set_title('Standard Initialization (0.01)')
    axes[2].set_xlabel('Weight Value')
    
    plt.tight_layout()
    plt.show()
    print("=" * 60)


# Run demonstrations
demonstrate_fp16_underflow()
gradient_clipping_demo()
initialization_comparison()

### Loss Scaling for Mixed Precision Training

In [ ]:
class MixedPrecisionTrainer:
    """
    Simplified mixed precision training with dynamic loss scaling.
    """
    
    def __init__(
        self,
        model: nn.Module,
        optimizer,
        initial_scale: float = 2**16,
        growth_factor: float = 2.0,
        backoff_factor: float = 0.5,
        growth_interval: int = 2000
    ):
        """
        Initialize mixed precision trainer.
        
        Args:
            model: PyTorch model
            optimizer: Optimizer instance
            initial_scale: Initial loss scale
            growth_factor: Factor to increase scale
            backoff_factor: Factor to decrease scale on overflow
            growth_interval: Steps between scale increases
        """
        self.model = model
        self.optimizer = optimizer
        self.loss_scale = initial_scale
        self.growth_factor = growth_factor
        self.backoff_factor = backoff_factor
        self.growth_interval = growth_interval
        self.steps_since_growth = 0
    
    def step(self, loss: torch.Tensor) -> bool:
        """
        Perform training step with loss scaling.
        
        Args:
            loss: Unscaled loss
        
        Returns:
            True if step succeeded, False if overflow detected
        """
        # Scale loss
        scaled_loss = loss * self.loss_scale
        
        # Backward pass
        scaled_loss.backward()
        
        # Check for gradient overflow/underflow
        has_inf_or_nan = False
        for param in self.model.parameters():
            if param.grad is not None:
                if torch.isnan(param.grad).any() or torch.isinf(param.grad).any():
                    has_inf_or_nan = True
                    break
        
        if has_inf_or_nan:
            # Overflow detected - reduce scale and skip update
            self.loss_scale *= self.backoff_factor
            self.steps_since_growth = 0
            self.optimizer.zero_grad()
            return False
        
        # Unscale gradients
        for param in self.model.parameters():
            if param.grad is not None:
                param.grad /= self.loss_scale
        
        # Update parameters
        self.optimizer.step()
        self.optimizer.zero_grad()
        
        # Grow scale if no overflow for growth_interval steps
        self.steps_since_growth += 1
        if self.steps_since_growth >= self.growth_interval:
            self.loss_scale *= self.growth_factor
            self.steps_since_growth = 0
        
        return True


# Demonstrate dynamic loss scaling
print("Dynamic Loss Scaling Demonstration")
print("=" * 60)

# Simulate training with varying gradient magnitudes
loss_scales = [2**16]  # Initial scale
steps_history = []
gradient_magnitudes = []

current_scale = 2**16
steps_since_growth = 0
growth_interval = 5

for step in range(100):
    # Simulate varying gradient magnitude
    # Occasionally create "overflow" scenario
    if step % 20 == 19:
        grad_magnitude = 1e10  # Overflow
    else:
        grad_magnitude = np.random.uniform(1e-4, 1e-2)
    
    gradient_magnitudes.append(grad_magnitude)
    steps_history.append(step)
    
    # Simulate overflow detection
    scaled_grad = grad_magnitude * current_scale
    has_overflow = scaled_grad > 65504  # FP16 max
    
    if has_overflow:
        # Reduce scale
        current_scale *= 0.5
        steps_since_growth = 0
    else:
        # Successful step
        steps_since_growth += 1
        if steps_since_growth >= growth_interval:
            current_scale *= 2.0
            steps_since_growth = 0
    
    loss_scales.append(current_scale)

# Visualize dynamic scaling
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

ax1.plot(steps_history, gradient_magnitudes, linewidth=1.5)
ax1.set_ylabel('Gradient Magnitude')
ax1.set_title('Gradient Magnitude Over Training')
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)
ax1.axhline(y=65504/2**16, color='r', linestyle='--', alpha=0.5, label='Overflow threshold')
ax1.legend()

ax2.plot(range(len(loss_scales)), loss_scales, linewidth=2, color='orange')
ax2.set_xlabel('Training Step')
ax2.set_ylabel('Loss Scale')
ax2.set_title('Dynamic Loss Scale Adjustment')
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Initial loss scale: {2**16}")
print(f"Final loss scale: {loss_scales[-1]:.0f}")
print(f"Number of scale adjustments: {len(set(loss_scales))}")
print("=" * 60)

## 8. Summary and Key Takeaways

This notebook demonstrated the mathematical foundations essential for understanding and training Large Language Models:

### Probability and Information Theory
- **Entropy** quantifies uncertainty in probability distributions
- **Cross-entropy loss** is the standard training objective for language models
- **KL divergence** measures the difference between distributions
- Lower entropy indicates more confident/peaked predictions

### Optimization
- **Adam optimizer** adapts learning rates per parameter using moment estimates
- **Momentum** accelerates convergence by accumulating gradient history
- **Learning rate schedules** (warmup + decay) are critical for stable training
- Gradient clipping prevents exploding gradients in deep networks

### Backpropagation and Architecture
- **Layer normalization** stabilizes gradient flow in deep transformers
- **Residual connections** provide direct gradient pathways
- **Attention mechanisms** have well-behaved gradients due to softmax normalization
- Proper initialization (Xavier, He) prevents vanishing/exploding activations

### Scaling Laws
- Model performance follows **predictable power laws** with respect to:
  - Number of parameters: L ∝ N^(-0.076)
  - Dataset size: L ∝ D^(-0.095)
  - Training compute: L ∝ C^(-0.05)
- **Chinchilla scaling**: Optimal training scales model size and data equally
- Memory requirements scale linearly with parameters
- Compute requirements scale quadratically with sequence length

### Numerical Stability
- **Mixed precision training** (FP16/FP32) reduces memory and increases speed
- **Dynamic loss scaling** prevents gradient underflow in FP16
- **Gradient clipping** prevents overflow and stabilizes training
- Proper initialization is crucial for stable training

These mathematical principles form the foundation for effective LLM training at scale.

## References

- Kingma & Ba (2014). "Adam: A Method for Stochastic Optimization" - [arXiv:1412.6980](https://arxiv.org/abs/1412.6980)
- Kaplan et al. (2020). "Scaling Laws for Neural Language Models" - [arXiv:2001.08361](https://arxiv.org/abs/2001.08361)
- Hoffmann et al. (2022). "Training Compute-Optimal Large Language Models" - [arXiv:2203.15556](https://arxiv.org/abs/2203.15556)
- Ba et al. (2016). "Layer Normalization" - [arXiv:1607.06450](https://arxiv.org/abs/1607.06450)
- Vaswani et al. (2017). "Attention Is All You Need" - [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
- Micikevicius et al. (2017). "Mixed Precision Training" - [arXiv:1710.03740](https://arxiv.org/abs/1710.03740)

For comprehensive theoretical treatment, see the [Mathematical Foundations chapter](../index.md).